In [ ]:
import os
import gc
import glob
from obspy import read, Stream

# ==============================================================================
# 🎛️ PARAMETER JALUR DATA LOKAL (Sesuaikan Folder SSD Mac Anda)
# ==============================================================================
# Input: Folder tempat penyimpanan berkas hasil unduhan gabungan terbaru Bapak (-15s s.d +120s)
INPUT_GABUNGAN_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'

# Output 1: Wadah penyimpanan sampel DATA GEMPA Tri-Komponen (7 Detik)
OUTPUT_EVENT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/dataset_ai/event_3c_7s'

# Output 2: Wadah penyimpanan sampel DATA NOISE Tri-Komponen (7 Detik)
OUTPUT_NOISE_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/dataset_ai/noise_3c_7s'

# Standarisasi Dimensi Mengikuti Metodologi Acuan Zhi Geng et al. (2025)
TARGET_SAMPLING_RATE = 100.0  # Resample ke 100 Hz
TARGET_DURATION_SEC = 7.0     # Jendela waktu wajib 7 detik penuh (700 Titik Sampel)

def process_and_split_waveform(file_path):
    try:
        st_master = read(file_path)
        
        # 🛡️ LANGKAH 1: Penyelarasan Frekuensi Digital ke 100 Hz untuk seluruh fasa komponen
        for tr in st_master:
            if tr.stats.sampling_rate != TARGET_SAMPLING_RATE:
                tr.resample(target_sampling_rate=TARGET_SAMPLING_RATE, no_filter=False)
                
        # Hitung titik waktu kedatangan gempa (Kompensasi offset mundur 15 detik dari hulu)
        t_arrival = st_master[0].stats.starttime + 15
        
        # ─── 📑 SEGMEN A: EKSTRAKSI DATA GEMPA 7 DETIK ───
        st_event = st_master.copy()
        event_start = t_arrival
        event_end = event_start + TARGET_DURATION_SEC
        st_event.trim(starttime=event_start, endtime=event_end, pad=False)
        
        for tr in st_event:
            expected_pts = int(TARGET_DURATION_SEC * TARGET_SAMPLING_RATE) # 700 Pts
            tr.data = tr.data[:expected_pts]
            tr.stats.npts = expected_pts
            
        # ─── 📑 SEGMEN B: EKSTRAKSI DATA NOISE STERIL 7 DETIK ───
        st_noise = st_master.copy()
        noise_start = t_arrival - 8  # Detik ke-8 sebelum gempa
        noise_end = t_arrival - 1    # Detik ke-1 sebelum gempa (Aman dari fasa P)
        st_noise.trim(starttime=noise_start, endtime=noise_end, pad=False)
        
        for tr in st_noise:
            expected_pts = int(TARGET_DURATION_SEC * TARGET_SAMPLING_RATE) # 700 Pts
            tr.data = tr.data[:expected_pts]
            tr.stats.npts = expected_pts

        # ─── 📦 LANGKAH WRITE FISIK KE DISK SSD MAC ───
        relative_path = os.path.relpath(file_path, INPUT_GABUNGAN_DIR)
        
        # Simpan file pecahan Gempa
        final_event_path = os.path.join(OUTPUT_EVENT_DIR, relative_path)
        os.makedirs(os.path.dirname(final_event_path), exist_ok=True)
        st_event.write(final_event_path, format="MSEED")
        
        # Simpan file pecahan Noise
        final_noise_path = os.path.join(OUTPUT_NOISE_DIR, relative_path)
        os.makedirs(os.path.dirname(final_noise_path), exist_ok=True)
        st_noise.write(final_noise_path, format="MSEED")
        
        del st_master, st_event, st_noise
        return True
    except Exception:
        return False

def run_splitting_pipeline():
    print("="*80)
    print("🚀 STARTING: WAVEFORM GABUNGAN ULTRA-FAST SPLITTER PIPELINE (7 SECONDS)")
    print("="*80)
    
    search_path = os.path.join(INPUT_GABUNGAN_DIR, "**", "*.mseed")
    all_files = glob.glob(search_path, recursive=True)
    total_files = len(all_files)
    print(f"📂 Mendata {total_files:,} berkas master gabungan di SSD untuk dipecah...")
    
    success_count = 0
    for file_path in all_files:
        if process_and_split_waveform(file_path):
            success_count += 1
            
    gc.collect()
    print("\n" + "="*60)
    print("🏁 PROSES PEMISAHAN DATASET SELESAI PARIPURNA")
    print("="*60)
    print(f"✅ Pasangan Dataset (Event & Noise) Sukses Diproduksi: {success_count:,} stasiun.")
    print("="*60)

if __name__ == "__main__":
    # run_splitting_pipeline()